In [ ]:
import copy
from IPython.display import display
import logging
import numpy
import os
import pandas
import pdb
import plotly
import pprint
import pyarrow
import pyarrow.parquet as pq
import six
import sys
import time

plotly.offline.init_notebook_mode(connected=True)

In [ ]:
LOG = logging.getLogger('CS230')
LOG.setLevel(logging.DEBUG)

# create a file handler
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)

# create a logging format
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', '%H-%M-%S')
handler.setFormatter(formatter)

# add the handlers to the logger
LOG.addHandler(handler)

In [ ]:
CWD = os.getcwd()
FILE_PATHS = []
EXCLUDE = ['grandsport.parquet', '250lm.parquet']

for dir_path, dir_names, file_names in os.walk(CWD):
    for file_name in file_names:
        if file_name.endswith('.parquet') and file_name not in EXCLUDE:
            file_path = os.path.join(dir_path, file_name)
            FILE_PATHS.append(file_path)
FILE_PATHS.sort()
LOG.debug(FILE_PATHS)

In [ ]:
#DATA_DIR = '2014_Targa_Sixty-Six'
#FILE_NAMES = [x for x in os.listdir(DATA_DIR) if (x != '250lm.parquet' and x.endswith('.parquet'))]
#FILE_NAMES.sort()
#LOG.debug(FILE_NAMES)

In [ ]:
COLUMNS_ORIG = ['time', 'handwheelAngle', 'throttle', 'brake', 'altitude', 'horizontalSpeed', 'vxCG', 'vyCG', 'yawAngle', 'pitchAngle', 'rollAngle', 'distance']  # 'latitude', 'longitude'

COLUMNS_TO_DIFF = ['yawAngle', 'pitchAngle', 'rollAngle', 'horizontalSpeed', 'distance', 'vxCG', 'vyCG']
COLUMN_DIFF_PREFIX = 'diff_'
COLUMNS = copy.deepcopy(COLUMNS_ORIG)
for column in COLUMNS_TO_DIFF:
    new_column = COLUMN_DIFF_PREFIX + column
    COLUMNS.append(new_column)

COLUMNS_WITH_GPS_JUMP = ['horizontalSpeed', 'vxCG', 'vyCG', 'yawAngle', 'pitchAngle', 'rollAngle']
DIFF_COLUMNS_WITH_GPS_JUMP = [COLUMN_DIFF_PREFIX + x for x in COLUMNS_WITH_GPS_JUMP]

STRIDES = [1]

# load data from parquet files

In [ ]:
DATA = {}

for file_path in FILE_PATHS[:5]:
    #file_path = os.path.join(DATA_DIR, file_name)
    file_name = file_path.replace(CWD + '/', '')
    
    # read parquet
    table = pq.read_table(file_path)
    
    # convert parquet table to pandas dataframe
    df = table.to_pandas()
    
    # save
    DATA[file_name] = {
        'orig': df
    }
    #sig_names = sorted(list(df.columns), key=lambda s: s.lower())
    
    LOG.debug('loaded %s', file_path)

# data size in rows

In [ ]:
data_size_total = 0

for file_name, dfs in six.iteritems(DATA):
    df = dfs['orig']
    LOG.debug('%s : %s' % (file_name, len(df)))
    data_size_total += len(df)

LOG.debug('total: %s', data_size_total)

# add columns for discrete derivatives

- need to correct diff_yawAngle for when wheel axis flips from positive to negative

In [ ]:
#fix_count = 0
#spot_check_freq = 1000

for file_name, dfs in six.iteritems(DATA):
    for stride in STRIDES:
        LOG.debug('%s stride: %s', file_name, stride)
        
        df = dfs['orig'].copy(deep=True)
        for column in COLUMNS_TO_DIFF:
            new_column = COLUMN_DIFF_PREFIX + column
            
            df[new_column] = df[column] - df[column].shift(stride)
        
        
        # correct for yawAngle sign flip
        indexes = df.index[(df['diff_yawAngle'] > 300) | (df['diff_yawAngle'] < -300)]
        for i in indexes:
            #fix_count += 1
            #if fix_count % spot_check_freq == 0:
            #    display(df.iloc[i-2:i+2][['yawAngle', 'diff_yawAngle']])

            #old_value = df.iloc[i]['diff_yawAngle']
            df.at[i, 'diff_yawAngle'] = (180 - abs(df.iloc[i]['yawAngle'])) + (180 - abs(df.iloc[i-stride]['yawAngle']))
            
            #if fix_count % spot_check_freq == 0:
            #    display(df.iloc[i-1:i+2][['yawAngle', 'diff_yawAngle']])
            #    LOG.warning('%s:%s fixed diff_yawAngle at %s (%s -> %s)', file_name, stride, i, round(old_value, 4), round(df.iloc[i]['diff_yawAngle'], 4))
        LOG.warning('# diff_yawAngle fixed: %s' % len(indexes))
        
        # replace NaN with zeros in diff columns
        values = {COLUMN_DIFF_PREFIX+x:0 for x in COLUMNS_TO_DIFF}
        df.fillna(value=values, inplace=True)
        
        DATA[file_name][stride] = df
    
    continue

### print max discrete derivative per column

In [ ]:
for file_name, dfs in six.iteritems(DATA):
    for stride in STRIDES:
        LOG.debug('%s stride: %s', file_name, stride)
        df = dfs[stride]
        for column in COLUMNS_TO_DIFF:
            diff_column = COLUMN_DIFF_PREFIX + column
            LOG.debug(' - %s: %s', diff_column, max(abs(df[diff_column])))

# clean discontinuities

find diff values with unexpected jump, clean if prior row had NaN

In [ ]:
thresholds = [
    ('diff_yawAngle', 10),
    ('diff_pitchAngle', 2),
    ('diff_rollAngle', 2),
    ('diff_distance', 10),
    ('diff_vxCG', 10),
    ('diff_vyCG', 10)
]

for file_name, dfs in six.iteritems(DATA):
    LOG.info(file_name)
    for stride in STRIDES:
        LOG.info('stride: %s', stride)
        df = dfs[stride]
        
        for column, threshold in thresholds:
            indexes = df.index[(df[column] > threshold) | (df[column] < -threshold)]
            for i in indexes:
                if numpy.isnan(df.iloc[i-stride]['altitude']):
                    LOG.warning('GPS NaN at %s : %s -> %s', i, 
                                df.iloc[i][DIFF_COLUMNS_WITH_GPS_JUMP].to_string(header=False, index=False).replace(os.linesep, ', '), 
                                df.iloc[i-1][DIFF_COLUMNS_WITH_GPS_JUMP].to_string(header=False, index=False).replace(os.linesep, ', '))
                    df.at[i, DIFF_COLUMNS_WITH_GPS_JUMP] = df.iloc[i-stride][DIFF_COLUMNS_WITH_GPS_JUMP]

### print max discrete derivative per column after cleaning

In [ ]:
for file_name, dfs in six.iteritems(DATA):
    for stride in STRIDES:
        LOG.debug('%s stride: %s', file_name, stride)
        df = dfs[stride]
        for column in COLUMNS_TO_DIFF:
            diff_column = COLUMN_DIFF_PREFIX + column
            LOG.debug(' - %s: %s', diff_column, round(max(abs(df[diff_column])), 4))

### inspect discontinuities

In [ ]:
manually_inspect = False

thresholds = [
    ('diff_yawAngle', 10),
    ('diff_pitchAngle', 2),
    ('diff_rollAngle', 2),
    ('diff_distance', 10),
    ('diff_vxCG', 10),
    ('diff_vyCG', 10)
]
discontinuities = {
    'count': {
        'total': 0
    },
    'indexes': {}
}

for file_name, dfs in six.iteritems(DATA):
    LOG.info(file_name)
    discontinuities['count'][file_name] = {}
    discontinuities['indexes'][file_name] = {}
    
    for stride in STRIDES:
        LOG.info('stride: %s', stride)
        df = dfs[stride]
        discontinuities['count'][file_name][stride] = 0
        discontinuities['indexes'][file_name][stride] = []
        
        for column, threshold in thresholds:
            indexes = df.index[(df[column] > threshold) | (df[column] < -threshold)]
            indexes_list = indexes.to_list()
            
            if manually_inspect:
                for i in indexes:
                    LOG.debug('%s : %s : %s', column, i, df.iloc[i][column])
                    display(df.iloc[i-5:i+5,:][COLUMNS])

                    pdb.set_trace()
            
            discontinuities['count']['total'] += len(indexes_list)
            discontinuities['count'][file_name][stride] += len(indexes_list)
            discontinuities['indexes'][file_name][stride].extend(indexes_list)

pprint.pprint(discontinuities['count'])

# view data tables

In [ ]:
for file_name, dfs in six.iteritems(DATA):
    for stride, df in six.iteritems(dfs):
        print('%s : %s : %s' % (file_name, stride, len(df)))
        if stride == 'orig':
            columns = COLUMNS_ORIG
        else:
            columns = COLUMNS
        display(df[columns].head())
        display(df[columns].tail())

# plot data

In [ ]:
for file_name, dfs in six.iteritems(DATA):
    LOG.debug('%s : %s', file_name, len(dfs[1]))

In [ ]:
num_points = 2000000
stride = 2500
print('max points per plot: %s' % (num_points / stride))

file_names = list(DATA.keys())

data = {}
layouts = {}
figs = {}
i = 0

for file_name in file_names:
    df = DATA[file_name][1]
    
    data[file_name] = []
    times = df['time'].values

    for sig_name in COLUMNS_TO_DIFF:
        trace = plotly.graph_objs.Scatter(
            x = times[:num_points:stride],
            y = df[COLUMN_DIFF_PREFIX + sig_name].values[:num_points:stride],
            name = COLUMN_DIFF_PREFIX + sig_name,
        )
        
        data[file_name].append(trace)
        
    layouts[file_name] = plotly.graph_objs.Layout(
        title=file_name
    )
    
    figs[file_name] = plotly.graph_objs.Figure(data=data[file_name], layout=layouts[file_name])
    plotly.offline.iplot(figs[file_name])

# plot as subplots

In [ ]:
num_points = 2000000
stride = 2500
print('max points per plot: %s' % (num_points / stride))

file_names = list(DATA.keys())

fig = plotly.tools.make_subplots(rows=len(DATA), cols=1, subplot_titles=file_names)
fig['layout'].update(height=3000, width=800)

data = {}
layouts = {}
i = 0

for file_name in file_names:
    df = DATA[file_name][1]
    i += 1
    
    data[file_name] = []
    times = df['time'].values

    for sig_name in COLUMNS_TO_DIFF:
        trace = plotly.graph_objs.Scatter(
            x = times[:num_points:stride],
            y = df[COLUMN_DIFF_PREFIX + sig_name].values[:num_points:stride],
            name = COLUMN_DIFF_PREFIX + sig_name,
        )
        
        fig.append_trace(trace, i, 1)

plotly.offline.iplot(fig)